# BioHub Cell Tracking During Development
## Chapter 46 — Division Dataset Expansion and Pipeline Audit

**Purpose:** Determine where ground-truth cell divisions fail in the current:

**Ch34 broad detections → Ch36 filtering → Ch37 refined tracks → Ch38 division stage**

This chapter generalizes the diagnostic logic from Chapters 42–45 from one known division to every auditable ground-truth division in the available training samples.

### Research question

> Across the available training data, where do true cell divisions actually fail in the current pipeline?

### Important constraint

This is a **diagnostic notebook**, not a model-improvement notebook. It does not tune filtering, rescue daughters, split tracks with GT, or train a new division classifier.

The notebook only reports what the current artifacts support. If Ch34/36/37 artifacts are available for only one sample, the audit will run on that sample and state that limitation. If Ch38 division-candidate artifacts are not available, the downstream candidate-generation/selection fields remain `NaN`/`AMBIGUOUS` rather than being invented.

In [1]:
!pip install -q zarr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 7.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 73.9 MB/s eta 0:00:00:00:0100:01


In [2]:
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import json
import math
import re
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zarr

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

## 1. Configuration

The physical scaling and diagnostic sample below preserve the conventions used in Chapter 42:

- `z_scale = 1.625`
- `y_scale = 0.40625`
- `x_scale = 0.40625`
- known diagnostic sample: `44b6_12dfb391`
- known diagnostic division: parent at `t=66`, daughters at `t=67`

Chapter 42 used both an 8-unit close radius and a 15-unit loose radius. Chapter 46 retains both distances and uses the **15-unit radius as the audit's "adequate candidate" threshold**, while preserving raw distances for later re-analysis.

In [3]:
@dataclass(frozen=True)
class Config:
    competition_dir: Path = Path(
        "/kaggle/input/competitions/biohub-cell-tracking-during-development"
    )

    data34_dir: Path = Path("/kaggle/input/datasets/emailchrismathews/data34")
    data36_dir: Path = Path("/kaggle/input/datasets/emailchrismathews/data36")
    data37_dir: Path = Path("/kaggle/input/datasets/emailchrismathews/data37")

    # Optional. Chapter 46 will use it only if a compatible Ch38 artifact exists.
    data38_dir: Path = Path("/kaggle/input/datasets/emailchrismathews/data38")

    output_dir: Path = Path("/kaggle/working")

    default_sample_id: str = "44b6_12dfb391"

    z_scale: float = 1.625
    y_scale: float = 0.40625
    x_scale: float = 0.40625

    strong_match_radius_physical: float = 8.0
    match_radius_physical: float = 15.0

    temporal_offsets: Tuple[int, ...] = (-1, 0, 1)

CONFIG = Config()
CONFIG.output_dir.mkdir(parents=True, exist_ok=True)

print(CONFIG)

Config(competition_dir=PosixPath('/kaggle/input/competitions/biohub-cell-tracking-during-development'), data34_dir=PosixPath('/kaggle/input/datasets/emailchrismathews/data34'), data36_dir=PosixPath('/kaggle/input/datasets/emailchrismathews/data36'), data37_dir=PosixPath('/kaggle/input/datasets/emailchrismathews/data37'), data38_dir=PosixPath('/kaggle/input/datasets/emailchrismathews/data38'), output_dir=PosixPath('/kaggle/working'), default_sample_id='44b6_12dfb391', z_scale=1.625, y_scale=0.40625, x_scale=0.40625, strong_match_radius_physical=8.0, match_radius_physical=15.0, temporal_offsets=(-1, 0, 1))


## 2. Discover the saved pipeline artifacts

The historical Chapter 42 audit used these files:

- `chapter34_top200_detections.csv`
- `chapter36_filtered_detections.csv`
- `chapter37_refined_track_nodes.csv`

This notebook searches recursively so it also works if Kaggle nests those files one directory deeper.

### Multi-sample rule

If a stage table contains a `sample_id` column, Chapter 46 audits every sample shared by Ch34/36/37 and the competition GT.

If the historical CSVs contain no `sample_id`, the notebook cannot safely pretend they represent multiple samples. In that case it assigns them only to the known Chapter 42 sample `44b6_12dfb391` and prints a warning.

In [ ]:
def find_csv(root: Path, exact_name: str, contains: Optional[str] = None) -> Optional[Path]:
    if not root.exists():
        return None

    exact = list(root.rglob(exact_name))
    if exact:
        return exact[0]

    if contains:
        fuzzy = [p for p in root.rglob("*.csv") if contains.lower() in p.name.lower()]
        if fuzzy:
            return fuzzy[0]

    return None


stage_paths = {
    "ch34": find_csv(
        CONFIG.data34_dir,
        "chapter34_top200_detections.csv",
        contains="top200",
    ),
    "ch36": find_csv(
        CONFIG.data36_dir,
        "chapter36_filtered_detections.csv",
        contains="filtered",
    ),
    "ch37": find_csv(
        CONFIG.data37_dir,
        "chapter37_refined_track_nodes.csv",
        contains="refined_track_nodes",
    ),
}

print("Discovered stage files:")
for stage, path in stage_paths.items():
    print(f"  {stage}: {path}")

missing_required = [stage for stage, path in stage_paths.items() if path is None]
if missing_required:
    raise FileNotFoundError(
        "Missing required pipeline artifacts: "
        + ", ".join(missing_required)
        + "\nAttach the Kaggle datasets used by Chapters 34, 36, and 37."
    )

In [ ]:
def normalize_stage_table(df: pd.DataFrame, stage: str) -> pd.DataFrame:
    df = df.copy()

    # Normalize common coordinate/time aliases without overwriting existing canonical columns.
    aliases = {
        "frame": "t",
        "time": "t",
        "timepoint": "t",
        "z_px": "z",
        "y_px": "y",
        "x_px": "x",
    }
    for old, new in aliases.items():
        if new not in df.columns and old in df.columns:
            df[new] = df[old]

    required = {"t", "z", "y", "x"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"{stage} is missing required columns {sorted(missing)}. "
            f"Available columns: {list(df.columns)}"
        )

    df["t"] = pd.to_numeric(df["t"], errors="coerce").astype("Int64")
    for c in ["z", "y", "x"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


stage_tables = {}
for stage, path in stage_paths.items():
    df = pd.read_csv(path)
    df = normalize_stage_table(df, stage)
    stage_tables[stage] = df
    print(f"{stage}: {len(df):,} rows | columns={list(df.columns)}")

In [ ]:
def attach_sample_id_if_needed(
    df: pd.DataFrame,
    stage: str,
    fallback_sample_id: str
) -> pd.DataFrame:
    df = df.copy()

    if "sample_id" in df.columns:
        df["sample_id"] = df["sample_id"].astype(str)
        return df

    # The saved Ch34/36/37 artifacts used in Ch42 were single-sample tables.
    df["sample_id"] = fallback_sample_id
    warnings.warn(
        f"{stage} has no sample_id column. Treating the entire table as "
        f"sample {fallback_sample_id!r}, matching the historical Chapter 42 setup. "
        "This does NOT constitute a multi-sample audit."
    )
    return df


for stage in stage_tables:
    stage_tables[stage] = attach_sample_id_if_needed(
        stage_tables[stage], stage, CONFIG.default_sample_id
    )

stage_sample_sets = {
    stage: set(df["sample_id"].dropna().astype(str).unique())
    for stage, df in stage_tables.items()
}
stage_sample_sets

## 3. Load GEFF ground truth and discover divisions programmatically

A GT division is defined the same way Chapter 42 identified the known event: a parent node with **out-degree exactly 2**.

Events with more than two outgoing edges are reported separately rather than silently converted into binary divisions.

In [ ]:
def competition_geff_paths() -> Dict[str, Path]:
    train_dir = CONFIG.competition_dir / "train"
    if not train_dir.exists():
        raise FileNotFoundError(
            f"Competition train directory not found: {train_dir}\n"
            "Attach the BioHub Cell Tracking competition data to this notebook."
        )

    geff_paths = {}
    for p in train_dir.glob("*.geff"):
        geff_paths[p.stem] = p

    if not geff_paths:
        # Some Kaggle mounts may expose .geff as directories that glob normally,
        # but keep a recursive fallback.
        for p in train_dir.rglob("*.geff"):
            geff_paths[p.stem] = p

    return geff_paths


GEFF_PATHS = competition_geff_paths()
print(f"Competition GT samples discovered: {len(GEFF_PATHS)}")
print(sorted(GEFF_PATHS)[:10], "..." if len(GEFF_PATHS) > 10 else "")

In [ ]:
def load_geff_tables(geff_path: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    geff = zarr.open(geff_path, mode="r")

    nodes = pd.DataFrame({
        "gt_node_id": np.asarray(geff["nodes/ids"][:]),
        "t": np.asarray(geff["nodes/props/t/values"][:]).astype(int),
        "z": np.asarray(geff["nodes/props/z/values"][:]).astype(float),
        "y": np.asarray(geff["nodes/props/y/values"][:]).astype(float),
        "x": np.asarray(geff["nodes/props/x/values"][:]).astype(float),
    })

    edge_ids = np.asarray(geff["edges/ids"][:])
    edges = pd.DataFrame({
        "gt_source": edge_ids[:, 0],
        "gt_target": edge_ids[:, 1],
    })

    return nodes, edges


def discover_gt_divisions(
    sample_id: str,
    geff_path: Path
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    nodes, edges = load_geff_tables(geff_path)

    out_degree = edges.groupby("gt_source").size()
    binary_parents = out_degree[out_degree == 2].index.tolist()
    unusual_parents = out_degree[out_degree > 2].index.tolist()

    node_lookup = nodes.set_index("gt_node_id")
    rows = []

    for parent_id in binary_parents:
        daughter_ids = edges.loc[
            edges["gt_source"] == parent_id, "gt_target"
        ].tolist()

        if parent_id not in node_lookup.index:
            continue
        if any(d not in node_lookup.index for d in daughter_ids):
            continue

        parent = node_lookup.loc[parent_id]
        daughters = node_lookup.loc[daughter_ids].sort_values(
            ["t", "z", "y", "x"]
        )

        if len(daughters) != 2:
            continue

        d1 = daughters.iloc[0]
        d2 = daughters.iloc[1]

        rows.append({
            "sample_id": sample_id,
            "division_id": f"{sample_id}:{parent_id}",
            "parent_gt_id": parent_id,
            "parent_t": int(parent["t"]),
            "parent_z": float(parent["z"]),
            "parent_y": float(parent["y"]),
            "parent_x": float(parent["x"]),
            "daughter_a_gt_id": daughters.index[0],
            "daughter_a_t": int(d1["t"]),
            "daughter_a_z": float(d1["z"]),
            "daughter_a_y": float(d1["y"]),
            "daughter_a_x": float(d1["x"]),
            "daughter_b_gt_id": daughters.index[1],
            "daughter_b_t": int(d2["t"]),
            "daughter_b_z": float(d2["z"]),
            "daughter_b_y": float(d2["y"]),
            "daughter_b_x": float(d2["x"]),
        })

    unusual = pd.DataFrame({
        "sample_id": sample_id,
        "parent_gt_id": unusual_parents,
        "out_degree": [int(out_degree.loc[p]) for p in unusual_parents],
    })

    return pd.DataFrame(rows), unusual

In [ ]:
common_stage_samples = (
    stage_sample_sets["ch34"]
    & stage_sample_sets["ch36"]
    & stage_sample_sets["ch37"]
)
auditable_samples = sorted(common_stage_samples & set(GEFF_PATHS))

print("Samples represented in all Ch34/36/37 artifacts:", sorted(common_stage_samples))
print("Auditable samples with competition GT:", auditable_samples)

if not auditable_samples:
    raise RuntimeError(
        "No common sample IDs exist across Ch34, Ch36, Ch37, and the competition GT."
    )

if len(auditable_samples) == 1:
    warnings.warn(
        "Only one sample is currently auditable from the saved Ch34/36/37 artifacts. "
        "Chapter 46 will still run, but the multi-sample hypothesis cannot be tested "
        "until equivalent stage artifacts exist for additional samples."
    )

division_tables = []
unusual_tables = []

for sample_id in auditable_samples:
    divs, unusual = discover_gt_divisions(sample_id, GEFF_PATHS[sample_id])
    division_tables.append(divs)
    if len(unusual):
        unusual_tables.append(unusual)
    print(
        f"{sample_id}: {len(divs)} binary GT divisions"
        + (f", {len(unusual)} >2-daughter events flagged" if len(unusual) else "")
    )

gt_divisions = (
    pd.concat(division_tables, ignore_index=True)
    if division_tables else pd.DataFrame()
)
unusual_gt_events = (
    pd.concat(unusual_tables, ignore_index=True)
    if unusual_tables else pd.DataFrame()
)

print(f"\nTotal binary GT divisions discovered: {len(gt_divisions)}")
display(gt_divisions.head())

## 4. Physical-coordinate matching helpers

All comparisons use physical-coordinate Euclidean distance.

The notebook stores both:
- **exact-frame nearest distance**, and
- **best distance in the ±1-frame window**.

This is important because Chapter 43 showed that exact-frame matching can make a daughter look absent even when strong evidence appears one frame later.

In [ ]:
def physical_xyz(z: float, y: float, x: float) -> np.ndarray:
    return np.array([
        float(z) * CONFIG.z_scale,
        float(y) * CONFIG.y_scale,
        float(x) * CONFIG.x_scale,
    ], dtype=float)


def add_physical_coordinates(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["z_physical"] = out["z"].astype(float) * CONFIG.z_scale
    out["y_physical"] = out["y"].astype(float) * CONFIG.y_scale
    out["x_physical"] = out["x"].astype(float) * CONFIG.x_scale
    return out


for stage in stage_tables:
    stage_tables[stage] = add_physical_coordinates(stage_tables[stage])

In [ ]:
def nearest_candidate_at_t(
    df: pd.DataFrame,
    sample_id: str,
    t: int,
    z: float,
    y: float,
    x: float,
) -> Dict:
    subset = df[
        (df["sample_id"].astype(str) == str(sample_id))
        & (df["t"].astype("Int64") == int(t))
    ].copy()

    if subset.empty:
        return {
            "found": False,
            "distance": np.nan,
            "index": None,
            "row": None,
        }

    target = physical_xyz(z, y, x)
    pts = subset[["z_physical", "y_physical", "x_physical"]].to_numpy(float)
    distances = np.linalg.norm(pts - target[None, :], axis=1)
    best_pos = int(np.argmin(distances))
    row = subset.iloc[best_pos]

    return {
        "found": True,
        "distance": float(distances[best_pos]),
        "index": row.name,
        "row": row,
    }


def nearest_candidate_window(
    df: pd.DataFrame,
    sample_id: str,
    expected_t: int,
    z: float,
    y: float,
    x: float,
    offsets=CONFIG.temporal_offsets,
) -> Dict:
    records = []
    for dt in offsets:
        result = nearest_candidate_at_t(
            df, sample_id, int(expected_t) + int(dt), z, y, x
        )
        records.append({
            "dt": int(dt),
            "t": int(expected_t) + int(dt),
            **result,
        })

    valid = [r for r in records if r["found"] and np.isfinite(r["distance"])]
    if not valid:
        return {
            "found": False,
            "distance": np.nan,
            "dt": np.nan,
            "t": np.nan,
            "index": None,
            "row": None,
            "records": records,
        }

    best = min(valid, key=lambda r: r["distance"])
    return {**best, "records": records}

## 5. Optional Ch38 artifact discovery

Chapters 42–45 give us strong evidence for Ch34/36/37 behavior. The exact saved Ch38 candidate table is not guaranteed to be attached to this Kaggle notebook.

Chapter 46 therefore does **not** invent a replacement Ch38 scoring algorithm.

If a compatible CSV exists under `data38`, this notebook loads it and tries to identify track-ID and score/selection columns. Otherwise, Ch38-specific fields remain unknown and are marked `AMBIGUOUS` after the track-topology stage.

In [ ]:
def discover_ch38_artifact(root: Path) -> Optional[Path]:
    if not root.exists():
        return None

    candidates = list(root.rglob("*.csv"))
    if not candidates:
        return None

    preferred_tokens = [
        "division_candidate",
        "division_candidates",
        "selected_division",
        "lineage",
    ]

    for token in preferred_tokens:
        hits = [p for p in candidates if token in p.name.lower()]
        if hits:
            return hits[0]

    return None


ch38_path = discover_ch38_artifact(CONFIG.data38_dir)
ch38_df = None

if ch38_path is not None:
    ch38_df = pd.read_csv(ch38_path)
    if "sample_id" not in ch38_df.columns:
        ch38_df["sample_id"] = CONFIG.default_sample_id
        warnings.warn(
            "Ch38 artifact has no sample_id column; assigning the historical "
            f"sample {CONFIG.default_sample_id!r} only."
        )
    print("Optional Ch38 artifact:", ch38_path)
    print("Ch38 columns:", list(ch38_df.columns))
    display(ch38_df.head())
else:
    print(
        "No compatible Ch38 CSV artifact found. "
        "Ch38 candidate-generation/selection audit will remain unknown."
    )

In [ ]:
def first_existing(columns: List[str], candidates: List[str]) -> Optional[str]:
    lower_map = {c.lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    return None


CH38_SCHEMA = None
if ch38_df is not None:
    cols = list(ch38_df.columns)
    CH38_SCHEMA = {
        "parent_track": first_existing(
            cols,
            ["parent_track_id", "parent_track", "parent_id", "source_track_id"]
        ),
        "daughter_a_track": first_existing(
            cols,
            ["daughter_a_track_id", "daughter1_track_id", "daughter_track_id_1",
             "daughter_track_a", "child1_track_id"]
        ),
        "daughter_b_track": first_existing(
            cols,
            ["daughter_b_track_id", "daughter2_track_id", "daughter_track_id_2",
             "daughter_track_b", "child2_track_id"]
        ),
        "score": first_existing(
            cols,
            ["division_score", "score", "candidate_score", "probability", "pred_proba"]
        ),
        "selected": first_existing(
            cols,
            ["selected", "is_selected", "chosen", "accepted"]
        ),
    }

    print("Inferred Ch38 schema:")
    print(json.dumps(CH38_SCHEMA, indent=2))

    required_triplet = [
        CH38_SCHEMA["parent_track"],
        CH38_SCHEMA["daughter_a_track"],
        CH38_SCHEMA["daughter_b_track"],
    ]
    if any(v is None for v in required_triplet):
        warnings.warn(
            "Could not infer a complete parent/daughter track triplet from the "
            "Ch38 artifact. Candidate-generation fields will remain unknown."
        )

## 6. Audit one GT node across stages

For each parent and daughter, capture:

- exact-frame distance
- best ±1-frame distance
- best temporal offset
- whether a candidate is within the 15-unit audit radius
- useful candidate metadata such as `rank_position` or `pred_dist` when present

In [ ]:
def candidate_metadata(row: Optional[pd.Series], prefix: str) -> Dict:
    out = {}
    for col in [
        "rank_position", "rank", "pred_dist", "score", "probability",
        "log_response", "intensity"
    ]:
        if row is not None and col in row.index:
            out[f"{prefix}_{col}"] = row[col]
    return out


def audit_target(
    stage_df: pd.DataFrame,
    sample_id: str,
    expected_t: int,
    z: float,
    y: float,
    x: float,
    prefix: str,
) -> Dict:
    exact = nearest_candidate_at_t(
        stage_df, sample_id, expected_t, z, y, x
    )
    window = nearest_candidate_window(
        stage_df, sample_id, expected_t, z, y, x
    )

    out = {
        f"{prefix}_exact_distance": exact["distance"],
        f"{prefix}_best_window_distance": window["distance"],
        f"{prefix}_best_dt": window["dt"],
        f"{prefix}_best_t": window["t"],
        f"{prefix}_adequate_exact": bool(
            exact["found"]
            and np.isfinite(exact["distance"])
            and exact["distance"] <= CONFIG.match_radius_physical
        ),
        f"{prefix}_adequate_window": bool(
            window["found"]
            and np.isfinite(window["distance"])
            and window["distance"] <= CONFIG.match_radius_physical
        ),
        f"{prefix}_strong_window": bool(
            window["found"]
            and np.isfinite(window["distance"])
            and window["distance"] <= CONFIG.strong_match_radius_physical
        ),
    }

    out.update(candidate_metadata(window["row"], prefix))
    return out

## 7. Track-topology audit

The track audit asks a different question from spatial detection:

> Even if a daughter has a nearby Ch37 node, does the trajectory containing that node look like a plausible daughter branch?

A daughter-associated track is suspicious when it starts **before** the GT division. Chapter 44 exposed exactly this failure mode.

The notebook does not split or repair those tracks.

In [ ]:
def match_track(
    ch37: pd.DataFrame,
    sample_id: str,
    expected_t: int,
    z: float,
    y: float,
    x: float,
) -> Dict:
    window = nearest_candidate_window(
        ch37, sample_id, expected_t, z, y, x
    )

    if not window["found"] or window["row"] is None:
        return {
            "track_id": np.nan,
            "distance": np.nan,
            "match_t": np.nan,
            "dt": np.nan,
            "track_start_t": np.nan,
            "track_end_t": np.nan,
            "track_length": np.nan,
            "adequate": False,
        }

    row = window["row"]
    if "track_id" not in row.index:
        raise ValueError(
            "Ch37 table does not contain track_id, so topology cannot be audited."
        )

    track_id = row["track_id"]
    sample_tracks = ch37[
        ch37["sample_id"].astype(str) == str(sample_id)
    ]
    track_rows = sample_tracks[sample_tracks["track_id"] == track_id]

    return {
        "track_id": track_id,
        "distance": float(window["distance"]),
        "match_t": int(window["t"]),
        "dt": int(window["dt"]),
        "track_start_t": int(track_rows["t"].min()),
        "track_end_t": int(track_rows["t"].max()),
        "track_length": int(len(track_rows)),
        "adequate": bool(window["distance"] <= CONFIG.match_radius_physical),
    }

In [ ]:
def track_topology_for_division(div: pd.Series, ch37: pd.DataFrame) -> Dict:
    sample_id = str(div["sample_id"])
    parent = match_track(
        ch37, sample_id,
        int(div["parent_t"]),
        div["parent_z"], div["parent_y"], div["parent_x"]
    )
    da = match_track(
        ch37, sample_id,
        int(div["daughter_a_t"]),
        div["daughter_a_z"], div["daughter_a_y"], div["daughter_a_x"]
    )
    db = match_track(
        ch37, sample_id,
        int(div["daughter_b_t"]),
        div["daughter_b_z"], div["daughter_b_y"], div["daughter_b_x"]
    )

    division_t = min(int(div["daughter_a_t"]), int(div["daughter_b_t"]))

    def predates(track):
        return bool(
            track["adequate"]
            and np.isfinite(track["track_start_t"])
            and int(track["track_start_t"]) < division_t
        )

    da_predates = predates(da)
    db_predates = predates(db)

    daughters_share = bool(
        da["adequate"]
        and db["adequate"]
        and pd.notna(da["track_id"])
        and pd.notna(db["track_id"])
        and da["track_id"] == db["track_id"]
    )

    daughter_tracks_distinct_from_parent = bool(
        da["adequate"]
        and db["adequate"]
        and parent["adequate"]
        and da["track_id"] != parent["track_id"]
        and db["track_id"] != parent["track_id"]
    )

    topology_compatible = bool(
        parent["adequate"]
        and da["adequate"]
        and db["adequate"]
        and not da_predates
        and not db_predates
        and not daughters_share
        and daughter_tracks_distinct_from_parent
    )

    return {
        "parent_track_id": parent["track_id"],
        "parent_track_distance": parent["distance"],
        "parent_track_start_t": parent["track_start_t"],
        "parent_track_end_t": parent["track_end_t"],
        "parent_track_length": parent["track_length"],

        "daughter_a_track_id": da["track_id"],
        "daughter_a_track_distance": da["distance"],
        "daughter_a_track_start_t": da["track_start_t"],
        "daughter_a_track_end_t": da["track_end_t"],
        "daughter_a_track_length": da["track_length"],
        "daughter_a_track_predates_division": da_predates,

        "daughter_b_track_id": db["track_id"],
        "daughter_b_track_distance": db["distance"],
        "daughter_b_track_start_t": db["track_start_t"],
        "daughter_b_track_end_t": db["track_end_t"],
        "daughter_b_track_length": db["track_length"],
        "daughter_b_track_predates_division": db_predates,

        "daughters_share_track": daughters_share,
        "daughter_tracks_distinct_from_parent": daughter_tracks_distinct_from_parent,
        "track_topology_compatible": topology_compatible,
    }

## 8. Optional Ch38 candidate lookup

When a compatible Ch38 artifact is present, a GT division is mapped to the nearest Ch37 parent/daughter track IDs and searched in the candidate table.

Daughter order is treated as interchangeable.

If the artifact cannot support that lookup, values remain unknown instead of being inferred.

In [ ]:
def normalize_id_for_compare(value):
    if pd.isna(value):
        return None
    try:
        f = float(value)
        if f.is_integer():
            return int(f)
    except Exception:
        pass
    return str(value)


def audit_ch38_candidate(
    sample_id: str,
    parent_track_id,
    daughter_a_track_id,
    daughter_b_track_id,
) -> Dict:
    unknown = {
        "true_division_candidate_generated": np.nan,
        "true_candidate_score": np.nan,
        "true_candidate_rank": np.nan,
        "true_candidate_selected": np.nan,
    }

    if ch38_df is None or CH38_SCHEMA is None:
        return unknown

    pcol = CH38_SCHEMA["parent_track"]
    acol = CH38_SCHEMA["daughter_a_track"]
    bcol = CH38_SCHEMA["daughter_b_track"]

    if any(c is None for c in [pcol, acol, bcol]):
        return unknown

    if any(pd.isna(v) for v in [parent_track_id, daughter_a_track_id, daughter_b_track_id]):
        return unknown

    subset = ch38_df[ch38_df["sample_id"].astype(str) == str(sample_id)].copy()

    p = normalize_id_for_compare(parent_track_id)
    da = normalize_id_for_compare(daughter_a_track_id)
    db = normalize_id_for_compare(daughter_b_track_id)

    def norm_series(series):
        return series.map(normalize_id_for_compare)

    pm = norm_series(subset[pcol]) == p
    direct = (
        (norm_series(subset[acol]) == da)
        & (norm_series(subset[bcol]) == db)
    )
    swapped = (
        (norm_series(subset[acol]) == db)
        & (norm_series(subset[bcol]) == da)
    )

    hits = subset[pm & (direct | swapped)].copy()
    if hits.empty:
        return {
            **unknown,
            "true_division_candidate_generated": False,
        }

    score_col = CH38_SCHEMA["score"]
    selected_col = CH38_SCHEMA["selected"]

    if score_col is not None:
        hits = hits.sort_values(score_col, ascending=False)
        best = hits.iloc[0]
        score = best[score_col]

        # Rank only among this sample's candidate table.
        sample_ranked = subset.sort_values(score_col, ascending=False).reset_index(drop=True)
        sample_ranked["_audit_rank"] = np.arange(1, len(sample_ranked) + 1)
        key_index = best.name
        rank_match = sample_ranked[sample_ranked.index == key_index]
        # The reset index means original identity is not preserved safely;
        # calculate score-based rank instead.
        rank = int((pd.to_numeric(subset[score_col], errors="coerce") > float(score)).sum() + 1)
    else:
        best = hits.iloc[0]
        score = np.nan
        rank = np.nan

    if selected_col is not None:
        selected_raw = best[selected_col]
        if isinstance(selected_raw, str):
            selected = selected_raw.strip().lower() in {"1", "true", "yes", "selected"}
        else:
            selected = bool(selected_raw)
    else:
        selected = np.nan

    return {
        "true_division_candidate_generated": True,
        "true_candidate_score": score,
        "true_candidate_rank": rank,
        "true_candidate_selected": selected,
    }

## 9. Build one audit row per GT division

The primary-failure classification uses explicit precedence because one division can have several problems.

Precedence:

1. no parent candidate
2. no daughter A candidate
3. no daughter B candidate
4. daughter filtered
5. temporal localization
6. track topology
7. Ch38 candidate not generated
8. Ch38 candidate not selected
9. success
10. ambiguous

All individual measurements are retained even after the primary label is assigned.

In [ ]:
def assign_primary_failure(row: Dict) -> str:
    # Candidate existence is evaluated in the ±1-frame Ch34 window.
    if not bool(row["parent_ch34_adequate_window"]):
        return "NO_PARENT_CANDIDATE"
    if not bool(row["daughter_a_ch34_adequate_window"]):
        return "NO_DAUGHTER_A_CANDIDATE"
    if not bool(row["daughter_b_ch34_adequate_window"]):
        return "NO_DAUGHTER_B_CANDIDATE"

    # Filtering failure: Ch34 has adequate evidence but Ch36 does not.
    if (
        not bool(row["daughter_a_ch36_adequate_window"])
        or not bool(row["daughter_b_ch36_adequate_window"])
    ):
        return "DAUGHTER_FILTERED"

    # Temporal localization: a daughter's best adequate Ch34 evidence is not at dt=0,
    # especially relevant when exact-frame evidence is inadequate.
    temporal_issue = (
        (
            bool(row["daughter_a_ch34_adequate_window"])
            and int(row["daughter_a_ch34_best_dt"]) != 0
            and not bool(row["daughter_a_ch34_adequate_exact"])
        )
        or
        (
            bool(row["daughter_b_ch34_adequate_window"])
            and int(row["daughter_b_ch34_best_dt"]) != 0
            and not bool(row["daughter_b_ch34_adequate_exact"])
        )
    )
    if temporal_issue:
        return "TEMPORAL_LOCALIZATION"

    if not bool(row["track_topology_compatible"]):
        return "TRACK_TOPOLOGY"

    generated = row.get("true_division_candidate_generated", np.nan)
    selected = row.get("true_candidate_selected", np.nan)

    if pd.isna(generated):
        return "AMBIGUOUS"
    if generated is False or generated == False:
        return "DIVISION_CANDIDATE_NOT_GENERATED"

    if pd.isna(selected):
        return "AMBIGUOUS"
    if selected is False or selected == False:
        return "DIVISION_CANDIDATE_NOT_SELECTED"

    return "SUCCESS"

In [ ]:
audit_rows = []

for _, div in gt_divisions.iterrows():
    sample_id = str(div["sample_id"])

    row = dict(div)

    targets = {
        "parent": (
            int(div["parent_t"]),
            div["parent_z"], div["parent_y"], div["parent_x"],
        ),
        "daughter_a": (
            int(div["daughter_a_t"]),
            div["daughter_a_z"], div["daughter_a_y"], div["daughter_a_x"],
        ),
        "daughter_b": (
            int(div["daughter_b_t"]),
            div["daughter_b_z"], div["daughter_b_y"], div["daughter_b_x"],
        ),
    }

    for stage in ["ch34", "ch36", "ch37"]:
        stage_df = stage_tables[stage]
        for role, (t, z, y, x) in targets.items():
            row.update(
                audit_target(
                    stage_df,
                    sample_id,
                    t, z, y, x,
                    prefix=f"{role}_{stage}",
                )
            )

    topology = track_topology_for_division(div, stage_tables["ch37"])
    row.update(topology)

    row["daughter_a_temporally_displaced"] = bool(
        row["daughter_a_ch34_adequate_window"]
        and int(row["daughter_a_ch34_best_dt"]) != 0
    )
    row["daughter_b_temporally_displaced"] = bool(
        row["daughter_b_ch34_adequate_window"]
        and int(row["daughter_b_ch34_best_dt"]) != 0
    )

    row.update(
        audit_ch38_candidate(
            sample_id,
            row["parent_track_id"],
            row["daughter_a_track_id"],
            row["daughter_b_track_id"],
        )
    )

    row["primary_failure_stage"] = assign_primary_failure(row)
    row["notes"] = ""

    audit_rows.append(row)

audit_df = pd.DataFrame(audit_rows)

print(f"Audited divisions: {len(audit_df)}")
display(audit_df.head())

## 10. Sanity check — rediscover the known Chapter 42 division

The generalized audit must locate the historical event without hard-coding it into the audit logic:

- sample `44b6_12dfb391`
- parent `172000000050` at `t=66`
- daughter A `173000000050` at `t=67`
- daughter B `173000000051` at `t=67`

If the event is absent, or the distances are radically inconsistent with Chapters 42–44, stop before trusting the aggregate results.

In [ ]:
KNOWN_SAMPLE = "44b6_12dfb391"
KNOWN_PARENT = 172000000050
KNOWN_DAUGHTERS = {173000000050, 173000000051}

sanity = audit_df[
    (audit_df["sample_id"].astype(str) == KNOWN_SAMPLE)
    & (audit_df["parent_gt_id"].astype(str) == str(KNOWN_PARENT))
].copy()

print(f"Known Chapter 42 division matches found: {len(sanity)}")

if len(sanity) == 1:
    sanity_cols = [
        "sample_id",
        "parent_gt_id",
        "daughter_a_gt_id",
        "daughter_b_gt_id",
        "parent_t",
        "daughter_a_t",
        "daughter_b_t",
        "parent_ch34_exact_distance",
        "daughter_a_ch34_exact_distance",
        "daughter_b_ch34_exact_distance",
        "daughter_a_ch34_best_window_distance",
        "daughter_a_ch34_best_dt",
        "daughter_b_ch34_best_window_distance",
        "daughter_b_ch34_best_dt",
        "daughter_a_ch36_best_window_distance",
        "daughter_b_ch36_best_window_distance",
        "daughter_a_track_id",
        "daughter_a_track_start_t",
        "daughter_b_track_id",
        "daughter_b_track_start_t",
        "track_topology_compatible",
        "primary_failure_stage",
    ]
    display(sanity[sanity_cols].T)
else:
    warnings.warn(
        "The known Chapter 42 event was not recovered exactly once. "
        "Do not trust aggregate Chapter 46 conclusions until this discrepancy is resolved."
    )

In [ ]:
# Historical approximate values from the saved Chapter 42 execution.
# These are used only as a diagnostic comparison, not as pass/fail assertions,
# because regenerated upstream artifacts can differ.

historical_reference = {
    "parent_ch34_exact_distance": 13.626,
    "daughter_a_ch34_exact_distance": 9.992,
    "daughter_b_ch34_exact_distance": 15.179,
    "daughter_a_ch34_best_window_distance": 6.563,
    "daughter_a_ch34_best_dt": 1,
    "daughter_b_ch34_best_window_distance": 6.749,
    "daughter_b_ch34_best_dt": 1,
}

if len(sanity) == 1:
    s = sanity.iloc[0]
    comparison = []
    for metric, historical in historical_reference.items():
        current = s.get(metric, np.nan)
        comparison.append({
            "metric": metric,
            "historical_ch42_value": historical,
            "chapter46_value": current,
            "absolute_difference": (
                abs(float(current) - float(historical))
                if pd.notna(current) else np.nan
            ),
        })
    display(pd.DataFrame(comparison))

## 11. Aggregate results

No percentages below are hypothetical. They are calculated from the rows produced by this execution.

In [ ]:
division_counts_by_sample = (
    audit_df.groupby("sample_id")
    .agg(
        gt_divisions=("division_id", "size"),
        ambiguous=("primary_failure_stage", lambda s: int((s == "AMBIGUOUS").sum())),
    )
    .reset_index()
)
division_counts_by_sample["fully_classified"] = (
    division_counts_by_sample["gt_divisions"]
    - division_counts_by_sample["ambiguous"]
)
display(division_counts_by_sample)

In [ ]:
total = len(audit_df)

stage_metrics = [
    (
        "Parent + both daughters have adequate Ch34 candidates",
        (
            audit_df["parent_ch34_adequate_window"]
            & audit_df["daughter_a_ch34_adequate_window"]
            & audit_df["daughter_b_ch34_adequate_window"]
        )
    ),
    (
        "Both daughters survive Ch36",
        (
            audit_df["daughter_a_ch36_adequate_window"]
            & audit_df["daughter_b_ch36_adequate_window"]
        )
    ),
    (
        "Track topology compatible",
        audit_df["track_topology_compatible"].fillna(False),
    ),
]

if audit_df["true_division_candidate_generated"].notna().any():
    stage_metrics.append(
        (
            "Correct division candidate generated",
            audit_df["true_division_candidate_generated"].fillna(False).astype(bool),
        )
    )

if audit_df["true_candidate_selected"].notna().any():
    stage_metrics.append(
        (
            "Correct division selected",
            audit_df["true_candidate_selected"].fillna(False).astype(bool),
        )
    )

stage_survival = pd.DataFrame([
    {
        "stage": name,
        "divisions_reaching_stage": int(mask.sum()),
        "percent_of_total": (100.0 * float(mask.sum()) / total) if total else np.nan,
    }
    for name, mask in stage_metrics
])

display(stage_survival)

In [ ]:
failure_distribution = (
    audit_df["primary_failure_stage"]
    .value_counts(dropna=False)
    .rename_axis("failure_stage")
    .reset_index(name="count")
)
failure_distribution["percent"] = (
    100.0 * failure_distribution["count"] / len(audit_df)
    if len(audit_df) else np.nan
)

display(failure_distribution)

In [ ]:
temporal_records = []

for role in ["daughter_a", "daughter_b"]:
    for _, row in audit_df.iterrows():
        temporal_records.append({
            "sample_id": row["sample_id"],
            "division_id": row["division_id"],
            "daughter_role": role,
            "best_dt": row[f"{role}_ch34_best_dt"],
            "best_window_distance": row[f"{role}_ch34_best_window_distance"],
            "exact_distance": row[f"{role}_ch34_exact_distance"],
            "adequate_window": row[f"{role}_ch34_adequate_window"],
            "adequate_exact": row[f"{role}_ch34_adequate_exact"],
        })

temporal_df = pd.DataFrame(temporal_records)

temporal_offset_summary = (
    temporal_df.groupby("best_dt", dropna=False)
    .agg(
        daughter_events=("division_id", "size"),
        adequate_events=("adequate_window", "sum"),
        median_best_distance=("best_window_distance", "median"),
    )
    .reset_index()
)

display(temporal_offset_summary)

## 12. Diagnostic plots

If there are too few divisions, interpret the tables and individual events more heavily than the plots.

In [ ]:
if len(failure_distribution):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(failure_distribution["failure_stage"], failure_distribution["count"])
    ax.set_title("Chapter 46 — Primary Division Failure Stage")
    ax.set_ylabel("GT divisions")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
if len(stage_survival):
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(stage_survival["stage"], stage_survival["divisions_reaching_stage"])
    ax.set_title("Chapter 46 — Pipeline Stage Survival")
    ax.set_ylabel("GT divisions reaching stage")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
if len(temporal_df):
    offset_counts = temporal_df["best_dt"].value_counts(dropna=False).sort_index()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(offset_counts.index.astype(str), offset_counts.values)
    ax.set_title("Best Ch34 Daughter Evidence by Temporal Offset")
    ax.set_xlabel("dt relative to annotated daughter frame")
    ax.set_ylabel("daughter events")
    plt.tight_layout()
    plt.show()

In [ ]:
if len(audit_df):
    per_sample = pd.crosstab(
        audit_df["sample_id"],
        audit_df["primary_failure_stage"]
    )
    per_sample.plot(kind="bar", stacked=True, figsize=(10, 5))
    plt.title("Division Outcomes by Sample")
    plt.ylabel("GT divisions")
    plt.xlabel("sample")
    plt.tight_layout()
    plt.show()

## 13. Save Chapter 46 artifacts

In [ ]:
output_files = {
    "audit": CONFIG.output_dir / "chapter46_division_pipeline_audit.csv",
    "division_counts": CONFIG.output_dir / "chapter46_division_counts_by_sample.csv",
    "stage_survival": CONFIG.output_dir / "chapter46_stage_survival.csv",
    "failure_distribution": CONFIG.output_dir / "chapter46_primary_failure_distribution.csv",
    "temporal_summary": CONFIG.output_dir / "chapter46_temporal_offset_summary.csv",
}

audit_df.to_csv(output_files["audit"], index=False)
division_counts_by_sample.to_csv(output_files["division_counts"], index=False)
stage_survival.to_csv(output_files["stage_survival"], index=False)
failure_distribution.to_csv(output_files["failure_distribution"], index=False)
temporal_offset_summary.to_csv(output_files["temporal_summary"], index=False)

if len(unusual_gt_events):
    unusual_path = CONFIG.output_dir / "chapter46_unusual_gt_events.csv"
    unusual_gt_events.to_csv(unusual_path, index=False)
    output_files["unusual_gt_events"] = unusual_path

zip_path = CONFIG.output_dir / "chapter46_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in output_files.values():
        zf.write(path, arcname=path.name)

print("Saved:")
for path in output_files.values():
    print(" ", path)
print(" ", zip_path)

## 14. Evidence-driven Chapter 47 recommendation

The recommendation is generated from the measured `primary_failure_stage` distribution.

Because Ch38 artifacts may be absent, `AMBIGUOUS` can legitimately dominate after topology-compatible events. That is a data-availability limitation, not evidence that Ch38 succeeds or fails.

In [ ]:
def chapter47_recommendation(audit: pd.DataFrame) -> str:
    if audit.empty:
        return "No auditable GT divisions were available."

    counts = audit["primary_failure_stage"].value_counts()

    # Ignore SUCCESS/AMBIGUOUS when choosing a technical bottleneck.
    actionable = counts.drop(labels=["SUCCESS", "AMBIGUOUS"], errors="ignore")

    if actionable.empty:
        if counts.get("AMBIGUOUS", 0) > 0:
            return (
                "Attach/export a compatible Chapter 38 division-candidate artifact "
                "before choosing the next modeling experiment."
            )
        return (
            "No dominant failure was identified. Validate more samples before "
            "changing the pipeline."
        )

    dominant = actionable.index[0]
    mapping = {
        "NO_PARENT_CANDIDATE":
            "Improve multiscale candidate recall / parent-sensitive detection.",
        "NO_DAUGHTER_A_CANDIDATE":
            "Improve multiscale candidate recall with emphasis on daughter-cell detection.",
        "NO_DAUGHTER_B_CANDIDATE":
            "Improve multiscale candidate recall with emphasis on daughter-cell detection.",
        "DAUGHTER_FILTERED":
            "Build division-preserving or context-aware candidate filtering.",
        "TEMPORAL_LOCALIZATION":
            "Model division evidence over a temporal window instead of one exact frame.",
        "TRACK_TOPOLOGY":
            "Build division-aware tracking with explicit birth/branch hypotheses.",
        "DIVISION_CANDIDATE_NOT_GENERATED":
            "Improve parent/daughter division candidate generation.",
        "DIVISION_CANDIDATE_NOT_SELECTED":
            "Improve division scoring/learning using the expanded positive dataset.",
    }

    return (
        f"Dominant measured failure: {dominant} "
        f"({int(actionable.iloc[0])} divisions). "
        f"Recommended Chapter 47 direction: {mapping.get(dominant, 'Investigate this stage.')}"
    )


recommendation = chapter47_recommendation(audit_df)
print(recommendation)

## 15. Completion summary

Chapter 46 is complete when the run answers these questions with measured counts:

- How many GT divisions are available in the auditable samples?
- How often do parent and both daughters have adequate Ch34 candidates?
- How often are daughter candidates lost by Ch36?
- How often is the strongest daughter evidence displaced by ±1 frame?
- How often is Ch37 topology incompatible with a true division?
- When Ch38 artifacts are available, how often is the true division candidate generated and selected?
- Which failure stage should determine Chapter 47?

### Important interpretation rule

If only the historical single-sample Ch34/36/37 artifacts are attached, this run is a **single-sample pipeline audit**. It is still useful and should reproduce the Chapter 42 diagnostic event, but it does not yet establish how failures are distributed across the full training set.

The next step in that case is not to invent multi-sample results. It is to generate equivalent Ch34/36/37 artifacts for additional training samples and rerun this same notebook.